# AgriShield

End-to-end soil-risk pipeline: harmonise surveys, train a classifier, query live satellite + climate for a new site.

```
LUCAS + WoSIS  ->  data/agrishield_training.csv  ->  RandomForest
New lat/lon    ->  GEE (Sentinel-2, WorldClim, static soil)  ->  risk report
```

**Hard rule this notebook follows:** every column in `FEATURE_COLUMNS` (agrishield/config.py) must be obtainable live, from lat/lon alone, with no lab test. Lab-only chemistry (OC, N, P, K, EC, CEC, bulk density) is the TARGET, never a feature.

Run cells top to bottom. Enrichment (2b) auto-skips itself if the training table already has satellite/climate columns, so re-running this notebook top-to-bottom after the first full run will NOT repeat the slow GEE pass or touch your data.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agrishield.config import TRAINING_CSV, FEATURE_COLUMNS, EXAMPLE_SITE
from agrishield.dataset import build_training_csv
from agrishield.climate import enrich_training_csv
from agrishield.model import train, save_model, predict_proba
from agrishield.inference import live_features, live_model_inputs

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
print("root:", ROOT)
print("training table exists:", TRAINING_CSV.exists())

root: d:\Jupyter Lab\Agrishield AI
training table exists: True


## 2. Base training table -- LUCAS + WoSIS only

No APIs yet, pure local files. Fast (seconds, not minutes).

This cell NEVER rebuilds if `TRAINING_CSV` already exists on disk -- it just loads it. `build_training_csv()` only runs the first time, when there's nothing there yet. This is deliberate: `build_training_csv()` only knows how to produce the raw 38-column merge, so calling it again on an already-enriched file would silently throw away every satellite/climate column you already paid GEE quota for.

`sample_year` is kept as a lookup key for step 2b -- it decides which satellite image to fetch for each row. It is never a model feature; check `FEATURE_COLUMNS` above if you want to confirm.

In [2]:
if TRAINING_CSV.exists():
    df = pd.read_csv(TRAINING_CSV, low_memory=False)
    print("loaded existing training table (NOT rebuilt) -- rows:", len(df), " cols:", df.shape[1])
else:
    df = build_training_csv()
    print("built fresh training table -- rows:", len(df), " cols:", df.shape[1])
print(df["source"].value_counts())
display(df.head())

loaded existing training table (NOT rebuilt) -- rows: 250578  cols: 53
source
wosis         195011
lucas_2009     19791
lucas_2018     18983
lucas_2015     16793
Name: count, dtype: int64


,source,sample_id,latitude,longitude,country,continent,sample_year,ph_h2o,ph_cacl2,oc_gkg,oc_20_30_gkg,n_gkg,p_mgkg,k_mgkg,ec,caco3,caco3_20_30,ox_al,ox_fe,clay_pct,...,cec_ph7,totc_gkg,acidic,low_oc,soil_stress,tmean_c,temp_seasonality,precip_mm,precip_seasonality,slope_deg,B2,B3,B4,B5,B6,B7,B8,B11,B12,ndvi
0,lucas_2018,47862690,47.150238,16.134212,AT,Europe,2018.0,4.81,4.1,12.4,NaN,1.1,NaN,101.9,8.73,3.0,NaN,NaN,NaN,24.0,...,NaN,NaN,1.0,0.0,1.0,9.1,7385.0,757.0,37.0,16.680049,239.000000,408.000,276.0,605.000000,1707.000000,2077.000000,2142.00,1017.000,486.000000,0.771712
1,lucas_2018,47882704,47.274272,16.175359,AT,Europe,2018.0,4.93,4.1,16.7,NaN,1.3,NaN,51.2,5.06,1.0,NaN,NaN,NaN,20.0,...,NaN,NaN,1.0,0.0,1.0,8.8,7352.0,744.0,37.0,4.197045,253.000000,437.000,353.0,812.000000,2307.000000,2837.000000,2698.00,1455.000,715.000000,0.768600
2,lucas_2018,47982688,47.123260,16.289693,AT,Europe,2018.0,4.85,4.1,47.5,NaN,3.1,12.3,114.8,12.53,1.0,NaN,NaN,NaN,23.0,...,NaN,NaN,1.0,0.0,1.0,9.3,7360.0,727.0,36.0,3.946203,223.000000,440.000,317.0,681.000000,2095.000000,2471.000000,2596.00,1458.000,703.000000,0.782355
3,lucas_2018,48022702,47.245693,16.357506,AT,Europe,2018.0,5.80,5.5,28.1,NaN,2.0,NaN,165.8,21.10,3.0,NaN,NaN,NaN,26.0,...,NaN,NaN,0.0,0.0,0.0,9.1,7324.0,701.0,37.0,9.195145,355.882353,548.200,487.0,793.812500,1985.383333,2409.666667,2602.75,1460.700,828.958333,0.684764
4,lucas_2018,48062708,47.296372,16.416782,AT,Europe,2018.0,6.48,6.1,19.4,NaN,2.2,NaN,42.1,10.89,2.0,NaN,NaN,NaN,25.0,...,NaN,NaN,0.0,0.0,0.0,9.0,7321.0,687.0,37.0,10.184996,286.190476,450.725,305.0,741.153846,2025.750000,2366.050000,2626.50,1376.625,725.700000,0.791915


### 2a. Know your date coverage before spending GEE quota

Sentinel-2 (the satellite you're querying in 2b) only exists from 2015 onward. Rows with an older or missing `sample_year` will end up with satellite columns as `NaN` -- expected, not a bug. WorldClim and static soil/elevation don't depend on date, so they'll still fill in for almost every row.

In [3]:
has_year = df["sample_year"].notna()
print("missing sample_year:", df["sample_year"].isna().sum(),
      f"({df['sample_year'].isna().mean()*100:.1f}%)")
print("year < 2015 (pre-Sentinel-2):", (df.loc[has_year, "sample_year"] < 2015).sum())
print("year >= 2015 (real spectral data possible):", (df.loc[has_year, "sample_year"] >= 2015).sum())

missing sample_year: 94427 (37.7%)
year < 2015 (pre-Sentinel-2): 120226
year >= 2015 (real spectral data possible): 35925


## 2b. Attach satellite + climate + soil columns

Three things get attached to the SAME rows as new columns:
- **Sentinel-2 bands + NDVI** -- batched per `sample_year` (one composite image per year, not one API call per row). Only fills for rows with `sample_year >= 2016` (rows before Sentinel-2 launched are skipped entirely -- not a failure, just physically impossible).
- **Static soil texture + elevation** -- from OpenLandMap/SRTM. Fills for virtually every row with coordinates, and OVERWRITES the LUCAS lab-measured `clay_pct`/`sand_pct`/`silt_pct`/`elevation_m` so training and live inference read from the identical source.
- **WorldClim** -- climatology, fills for virtually every row.

This cell auto-detects whether enrichment already ran (checks for `B2`/`tmean_c` with real values) and skips the slow GEE pass if so -- safe to re-run top-to-bottom any time.

In [4]:
probe_cols = ["B2", "tmean_c"]
already_enriched = all(c in df.columns for c in probe_cols) and df[probe_cols].notna().any().all()

if already_enriched:
    print("training table already enriched -- skipping GEE calls")
else:
    df = enrich_training_csv(batch_size=400, max_rows=None)
    print("enrichment complete -- rows:", len(df), " cols:", df.shape[1])

training table already enriched -- skipping GEE calls


## 3. Train

Loads from disk (not the in-memory `df` above) so this cell works even if you restarted the kernel after the overnight run finished. This is the production model, trained on all rows (satellite-era and pre-satellite alike -- non-satellite rows just get median-imputed band values).

In [5]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")
model, report = train(df)
print(report)
save_model(model)

cols = [c for c in FEATURE_COLUMNS if c in df.columns]
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

training on 250578 rows
class balance (target = acidic):
acidic
0.0    0.694
1.0    0.306
              precision    recall  f1-score   support

         0.0      0.901     0.823     0.860     26652
         1.0      0.660     0.792     0.720     11604

    accuracy                          0.813     38256
   macro avg      0.781     0.808     0.790     38256
weighted avg      0.828     0.813     0.818     38256


feature importances:
clay_pct              0.140913
precip_mm             0.108991
B3                    0.082184
tmean_c               0.071869
B12                   0.064327
temp_seasonality      0.059903
precip_seasonality    0.057729
elevation_m           0.048297
slope_deg             0.040673
B5                    0.039659
sand_pct              0.039282
silt_pct              0.035159
B2                    0.034727
ndvi                  0.031110
B11                   0.031017
B4                    0.030723
B6                    0.029024
B8                    0.027759
B7 

### 3b. Optional comparison: satellite-era rows only (`sample_year >= 2015`)

Same model, trained on just the rows that have REAL (not imputed) spectral bands, so you can see whether accuracy actually improves on cleaner data or whether the extra pre-satellite rows were pulling their weight.

In [6]:
df_recent = df[df["sample_year"] >= 2015].copy()
print("rows with real satellite era data:", len(df_recent))
model_recent, report_recent = train(df_recent)
print("--- full dataset ---")
print(report)
print("--- satellite-era only ---")
print(report_recent)

rows with real satellite era data: 35925
class balance (target = acidic):
acidic
0.0    0.652
1.0    0.348
--- full dataset ---
              precision    recall  f1-score   support

         0.0      0.901     0.823     0.860     26652
         1.0      0.660     0.792     0.720     11604

    accuracy                          0.813     38256
   macro avg      0.781     0.808     0.790     38256
weighted avg      0.828     0.813     0.818     38256

--- satellite-era only ---
              precision    recall  f1-score   support

         0.0      0.884     0.877     0.880      4572
         1.0      0.787     0.797     0.792      2603

    accuracy                          0.848      7175
   macro avg      0.835     0.837     0.836      7175
weighted avg      0.848     0.848     0.848      7175



## 4. Live inference for a new site

This is what actually runs when a farmer taps Calculate -- one coordinate, live GEE + weather calls, no training data touched. `live_model_inputs` feeds the model; `live_features` wraps that plus current weather for a human-facing report.

In [7]:
site = EXAMPLE_SITE
report_data = live_features(site["latitude"], site["longitude"])
print(site["name"], site["latitude"], site["longitude"])
report_data

Clayton, Victoria -37.91 145.13


{'latitude': -37.91,
 'longitude': 145.13,
 'model_inputs': {'B2': 296,
  'B3': 453.5,
  'B4': 452.5,
  'B5': 1122.5,
  'B6': 2031,
  'B7': 2326,
  'B8': 2344.5,
  'B11': 1574,
  'B12': 1179,
  'ndvi': 0.6764390468597412,
  'elevation_m': 101,
  'slope_deg': 0.3743423819541931,
  'clay_pct': 16,
  'sand_pct': 70,
  'silt_pct': 14.0,
  'tmean_c': 14.4,
  'temp_seasonality': 3660,
  'precip_mm': 836,
  'precip_seasonality': 17},
 'weather_now': {'temp_c': 17.73,
  'humidity': 68,
  'pressure': 1017,
  'wind_speed': 0.89,
  'rain_1h_mm': 0,
  'description': 'overcast clouds'}}

In [8]:
inputs = live_model_inputs(site["latitude"], site["longitude"])
print(inputs)
predict_proba(model, inputs)

{'B2': 296, 'B3': 453.5, 'B4': 452.5, 'B5': 1122.5, 'B6': 2031, 'B7': 2326, 'B8': 2344.5, 'B11': 1574, 'B12': 1179, 'ndvi': 0.6764390468597412, 'elevation_m': 101, 'slope_deg': 0.3743423819541931, 'clay_pct': 16, 'sand_pct': 70, 'silt_pct': 14.0, 'tmean_c': 14.4, 'temp_seasonality': 3660, 'precip_mm': 836, 'precip_seasonality': 17}


{'predicted': 1, 'probability': {0: 0.3353536128997803, 1: 0.6646463871002197}}

## 5. Diagnostics

Not part of the production path -- these cells check the training table and the model's behaviour before you trust it. Keep them; they're what catch a silently-broken enrichment run or a model that's leaning on geography instead of soil physics.

In [9]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("total rows:", len(df))

sentinel_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]
static_cols   = ["clay_pct","sand_pct","silt_pct","elevation_m","slope_deg"]
climate_cols  = ["tmean_c","precip_mm","temp_seasonality","precip_seasonality"]

print("\n--- coverage (fraction non-null) ---")
print(df[sentinel_cols + static_cols + climate_cols].notna().mean().round(3).to_string())

print("\nrows with at least one real sentinel band:", df[sentinel_cols].notna().any(axis=1).sum())
print("rows with sample_year >= 2016:", (df["sample_year"] >= 2016).sum())

total rows: 250578

--- coverage (fraction non-null) ---
B2                    0.076
B3                    0.076
B4                    0.076
B5                    0.076
B6                    0.076
B7                    0.076
B8                    0.076
B11                   0.076
B12                   0.076
ndvi                  0.076
clay_pct              0.993
sand_pct              0.993
silt_pct              0.997
elevation_m           0.960
slope_deg             0.960
tmean_c               0.998
precip_mm             0.998
temp_seasonality      0.998
precip_seasonality    0.998

rows with at least one real sentinel band: 19091
rows with sample_year >= 2016: 19091


In [10]:
sat_subset = df[df["B2"].notna()].copy()
print("rows with real satellite data:", len(sat_subset))

sat_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]

# WITH satellite bands
model_with, report_with = train(sat_subset)

# WITHOUT -- same rows, just drop the band columns before training
sat_subset_no_bands = sat_subset.drop(columns=[c for c in sat_cols if c in sat_subset.columns])
model_without, report_without = train(sat_subset_no_bands)

print("--- WITH satellite bands ---")
print(report_with)
print("--- WITHOUT satellite bands (same rows) ---")
print(report_without)

rows with real satellite data: 19091
class balance (target = acidic):
acidic
0.0    0.68
1.0    0.32
class balance (target = acidic):
acidic
0.0    0.68
1.0    0.32
--- WITH satellite bands ---
              precision    recall  f1-score   support

         0.0      0.895     0.897     0.896      2561
         1.0      0.790     0.785     0.787      1257

    accuracy                          0.860      3818
   macro avg      0.842     0.841     0.842      3818
weighted avg      0.860     0.860     0.860      3818

--- WITHOUT satellite bands (same rows) ---
              precision    recall  f1-score   support

         0.0      0.879     0.871     0.875      2561
         1.0      0.742     0.757     0.749      1257

    accuracy                          0.833      3818
   macro avg      0.811     0.814     0.812      3818
weighted avg      0.834     0.833     0.834      3818



In [11]:
print(df["continent"].value_counts(dropna=False))

continent
Europe              91927
Northern America    67179
Oceania             32485
Africa              28598
South America       23434
Asia                 6920
Antarctica             35
Name: count, dtype: int64


## 6. Continent generalization check (production model)

XGBoost is already the production model -- `model.py`'s `train()` (used in section 3 above) trains tuned XGBoost internally, so cells 9/11/13/14/17 above were already using it with zero notebook changes needed. This section is the diagnostic: does it actually generalize across continents, or just look good in-distribution.

In [12]:
from agrishield.model import continent_holdout_check

continent_holdout_check(df)

c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Nihar\anaconda3\Lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B11' 'B12' 'ndvi']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



--- trained WITHOUT Europe, tested ON Europe (n=74127) ---
              precision    recall  f1-score   support

         0.0      0.774     0.749     0.761     48952
         1.0      0.541     0.576     0.558     25175

    accuracy                          0.690     74127
   macro avg      0.658     0.662     0.660     74127
weighted avg      0.695     0.690     0.692     74127


--- trained WITHOUT Africa, tested ON Africa (n=23248) ---
              precision    recall  f1-score   support

         0.0      0.845     0.751     0.795     17687
         1.0      0.415     0.563     0.478      5561

    accuracy                          0.706     23248
   macro avg      0.630     0.657     0.637     23248
weighted avg      0.742     0.706     0.719     23248


--- trained WITHOUT Northern America, tested ON Northern America (n=57115) ---
              precision    recall  f1-score   support

         0.0      0.785     0.830     0.807     41643
         1.0      0.460     0.389    